# Problem Setup
Here we use a neural network to approximate a function:
$$f(x)=sin(x)$$


# Libraries

In [ ]:
from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from pyDOE import lhs

from Lib.ImportLibrary import configure_notebook
from Lib.MultiLayerNet import MultiLayerNet

dev = configure_notebook()
output_dir = Path("output/Case1")
output_dir.mkdir(parents=True, exist_ok=True)
checkpoint = output_dir / "model.pt"

# Training parameters

In [ ]:
steps=20000
lr=1e-3
N = 500

# Functions

In [ ]:
def f(x):
    y=np.sin(x)
    return y 

# Generate data

In [ ]:
# get the analytical solution over the full domain
x = 2 * np.pi * lhs(1, N) # Latin Hypercube sampling for collocation points 
y = f(x)
print(x.shape, y.shape)


In [ ]:
fig, ax1 = plt.subplots()
ax1.scatter(x,y,color='blue',label='Real')
ax1.set_xlabel('x',color='black')
ax1.set_ylabel('f(x)',color='black')
ax1.tick_params(axis='y', color='black')
ax1.legend(loc = 'upper left')

# Train Neural Network

In [ ]:
x_train = torch.from_numpy(x).float().to(dev)
y_train = torch.from_numpy(y).float().to(dev)

# Set Neural Network
layers = [1,30,30,30,1]
model = MultiLayerNet(layers, init=True)
model.to(dev)
optimizer = torch.optim.Adam(model.parameters(),lr=lr,amsgrad=False)

In [ ]:
start_time = time.time()
it_time = time.time()
for i in range(steps):
    optimizer.zero_grad()
    yh = model(x_train)
    loss = torch.mean(torch.square(yh-y_train)) # use mean squared error 
    loss.backward()
    optimizer.step()
    if i % max(1, steps // 10) == 0:
      print('Iter: %d  Loss: %.5e  Time: %.3e'
                    % (i, loss.item(), time.time() - it_time))
      it_time = time.time()

torch.save(model.state_dict(), checkpoint)

# Test Model

In [ ]:
x_test = torch.linspace(0, 2 * np.pi, N, device=dev).view(-1,1)
y_pred = model(x_test)

x_test = x_test.detach().cpu().numpy()
y_pred = y_pred.detach().cpu().numpy()

y_real = f(x_test)

In [ ]:
fig, ax1 = plt.subplots()
ax1.plot(x_test,y_real,color='blue',label='Real')
ax1.plot(x_test,y_pred,':',color='red',label='Pred')
ax1.set_xlabel('x',color='black')
ax1.set_ylabel('f(x)',color='black')
ax1.tick_params(axis='y', color='black')
ax1.legend(loc = 'upper left')

In [ ]:
print('MSE:', np.mean(np.square(y_real-y_pred)))